# Ensemble Results Processing
This code processes ensemble results from multiple model members for extreme event classification.

## Key Features:
* Ensemble Generation: Creates 20 model variants using different random seeds

* Multi-site Processing: Analyzes data for multiple geographical locations

* Dual Input Architecture: Combines local features (NN) and spatial data (CNN)

* Flexible Configuration: Supports both SPEI/SPI drought indices and soil moisture data

* Comprehensive Outputs: Generates both performance metrics and SHAP explainability results

## Output Files:
* Performance Metrics: Accuracy scores and prediction probabilities

* XAI Results: Ensemble-averaged SHAP values with standard deviations

* Text Reports: Detailed accuracy summaries for each

In [ ]:

def generate_ensemble_seeds(fixed_seed=123):
    rng = np.random.default_rng(fixed_seed) #generator
    seeds = rng.integers(low=0, high=2**32 - 1, size=20).tolist()
    return seeds

def reset_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    g.manual_seed(seed)  # For DataLoader's generator

list_seeds = generate_ensemble_seeds(fixed_seed=123)

sites = ['cordoba', 'stockholm', 'lyon', 'hannover','belgrado'] # All sites!


#File paths large-scale data 
        
file_g500 = "/gpfs/scratch/bsc32/bsc167965/data/era5/lagged_anomalies/g500_1x1_lagged_standarized_anomalies.nc"
file_g200 = "/gpfs/scratch/bsc32/bsc167965/data/era5/lagged_anomalies/g200_1x1_lagged_standarized_anomalies.nc"
file_psl = "/gpfs/scratch/bsc32/bsc167965/data/era5/lagged_anomalies/psl_1x1_lagged_standarized_anomalies.nc"

# File CO2 data 
file_CO2 = "/home/bsc/bsc167965/TFM/ML/data_files/daily_co2_JJA.nc"

# File local-scale

# ============================================================================================================================
# General Configuration 
# ============================================================================================================================

# ========================================================================================
# Percentile used in extreme classification 
percentile = '90p'
# ========================================================================================

# ========================================================================================
# SPEI or SPI configuration
using_spei = False
scales_spei = ['30','60']
spei_spi = 'spei'
spei_spi_variable_mapping = {
    'spei': [f'spei_hg_{scale_spei}' for scale_spei in scales_spei],
    'spi': [f'spi_{scale_spei}' for scale_spei in scales_spei]
    }

spei_variables = spei_spi_variable_mapping[spei_spi] # Variable name in the dataset for SPEI
distribution = 'gamma'

# ========================================================================================

start_date = "1950-01-01"

variables_era5 = ['g500', 'g200', 'psl'] 
variables_era5land = ['swvl1','swvl2','swvl3']

if using_spei:
    _ERA5LAND_TRAIN_DATASET_CONF = dict(
        start_date= start_date,
        end_date= "2013-12-31",
        months = [6,7,8],
        )

    _ERA5LAND_TEST_DATASET_CONF = dict(
        start_date= "2014-01-01",
        end_date= "2023-12-31",
        months = [6,7,8],
        )

else:
    _ERA5LAND_TRAIN_DATASET_CONF = dict(
        start_date= start_date,
        end_date= "2013-12-31",
        months = [6,7,8],
        variables = variables_era5land
        )

    _ERA5LAND_TEST_DATASET_CONF = dict(
        start_date= "2014-01-01",
        end_date= "2023-12-31",
        months = [6,7,8],
        variables = variables_era5land
        )

_ERA5_TRAIN_DATASET_CONF = dict(
start_date= start_date,
end_date= "2013-12-31",
months = [6,7,8],
start_lag = 1,
lags_era5 = 1,
variables = variables_era5
    )

_ERA5_TEST_DATASET_CONF = dict(
start_date= "2014-01-01",
end_date= "2023-12-31",
months = [6,7,8],
start_lag = 1,
lags_era5 = 1,
variables = variables_era5
    )

number_lags = _ERA5_TRAIN_DATASET_CONF['lags_era5']

name_save_CombinedModel = f"CO2_Combinedmodel_trained_with_cnn_nn_trained_together_{number_lags}lags"


# Start loop -------------------------------------------------------------------------------

device = torch.device("cpu")

for site in sites:

    print(f"Doing site : {site}")

    # Local scale data files - SPEI or SPI
    if spei_spi == 'spi':
        files_spei =[f"/gpfs/projects/bsc32/bsc167965/observational_TX/{distribution}_{spei_spi}_tasmax_{site}_{scale_spei}_daily.nc"
                    for scale_spei in scales_spei] # For testing with 1 month scale
    elif spei_spi == 'spei':
        files_spei = [f"/gpfs/projects/bsc32/bsc167965/observational_TX/{distribution}_daily_{spei_spi}{scale_spei}_tasmax_{site}_hg.nc"
                        for scale_spei in scales_spei]

    # Load datasets --------------------------------------------------------------------------------------------------
    # Large-scale
    train_features_era5 = machine_learning.LargeScale_Dataset_extremes(file_g500,file_g200,file_psl, **_ERA5_TRAIN_DATASET_CONF) # shape: features, time, lat, lon 
    test_features_era5 = machine_learning.LargeScale_Dataset_extremes(file_g500,file_g200,file_psl, **_ERA5_TEST_DATASET_CONF)

    # Local-scale
    if using_spei:
        file_local_scale = f"/gpfs/scratch/bsc32/bsc167965/data/era5_land/obs_lagged_anomalies_and_event_detection/obs_{site}_lagged_standarized_anomalies_and_extreme_detection.nc"
        train_dataset = SPEI_extremes_location_dataset(file_path=file_local_scale, file_CO2=file_CO2 , files_spei = files_spei, **_ERA5LAND_TRAIN_DATASET_CONF, spei_variables = spei_variables, num_lags=7)
        test_dataset = SPEI_extremes_location_dataset(file_path=file_local_scale, file_CO2=file_CO2 , files_spei = files_spei, **_ERA5LAND_TEST_DATASET_CONF, spei_variables = spei_variables, num_lags=7)
    
    else:
        file_local_scale = f"/gpfs/scratch/bsc32/bsc167965/data/era5_land/lagged_anomalies_and_event_detection/{percentile}_{site}_lagged_standarized_anomalies_and_extreme_detection.nc"
        train_dataset = machine_learning.LocalScale_Dataset_extremes_location_swvl_averaged_including_CO2(file_path=f"/gpfs/scratch/bsc32/bsc167965/data/era5_land/lagged_anomalies_and_event_detection/{percentile}_{site}_lagged_standarized_anomalies_and_extreme_detection.nc", file_CO2=file_CO2 ,  **_ERA5LAND_TRAIN_DATASET_CONF)
        test_dataset = machine_learning.LocalScale_Dataset_extremes_location_swvl_averaged_including_CO2(file_path=f"/gpfs/scratch/bsc32/bsc167965/data/era5_land/lagged_anomalies_and_event_detection/{percentile}_{site}_lagged_standarized_anomalies_and_extreme_detection.nc", file_CO2=file_CO2 ,  **_ERA5LAND_TEST_DATASET_CONF)
        
    # Prepare for ensemble results --------------------------------------------------------------------------------------------------
    outputs_prob_seeds = np.zeros((len(list_seeds),test_dataset.features.shape[0],2))
    ensamble_shap_values_nn_raw = np.zeros((len(list_seeds),test_dataset.features.shape[0],test_dataset.features.shape[1]))
    ensamble_shap_values_cnn_raw = np.zeros((len(list_seeds),test_dataset.features.shape[0],test_features_era5.features.shape[1],58,124))
    
    # Loop over seeds to load models and get results --------------------------------------------------------------------------------
    
    for count_seed,seed in enumerate(list_seeds[:]):         
        
        g = torch.Generator()
        
        reset_seeds(seed)

        # Dataloader configurations --------------------------------------------------------------------------------
         
        _DATALOADERS_CONF = dict(
        batch_size= 32,
        drop_last= False,
        shuffle = True,
        num_workers=0,
        generator=g)
        
        _DATALOADERS_TEST_CONF = dict(
                batch_size= 32,
                drop_last= False,
                shuffle = False,
                num_workers=0)
        
        # Combined Dataset and Dataloader -------------------------------------------------------------------------
        
        batch_size = _DATALOADERS_CONF['batch_size'] # batch size for dataloaders both datasets
   
        combined_train_dataset  = machine_learning.CombinedDataset(train_dataset,train_features_era5, variables = variables_era5  )
        combined_test_dataset = machine_learning.CombinedDataset(test_dataset,test_features_era5, variables = variables_era5  )
        
        # Split train and validation
        train_size_combined = int(0.8 * len(combined_train_dataset))
        val_size_combined = len(combined_train_dataset) - train_size_combined

        combined_test_loader = DataLoader(combined_test_dataset, **_DATALOADERS_TEST_CONF)
    
        NN_model_loaded = machine_learning.ToCombineExtremeClassifier(input_dim=len(train_dataset.all_features), train_alone_NN=False, num_classes=2).to(device)
        NN_model_loaded.eval()
        reset_seeds(seed)
        CNN_model_loaded = convnext_functions.ConvNext(
               num_channels=len(train_features_era5.all_features),
               num_classes=2,
               patch_size=4,
               layer_dims=[4, 6,6,16],
               depths=[1, 2,2,1],
               drop_rate=0.05,
               train_alone=False,
        ).to(device) 
        reset_seeds(seed)
        
        model = machine_learning.CombinedModel(NN_model_loaded, CNN_model_loaded, nn_hidden_dim=8, cnn_hidden_dim=16,output_dim=2).to(device)
        reset_seeds(seed)
    
        main_path = f'/gpfs/scratch/bsc32/bsc167965/data/test_train_n_shap_dilation/{site}/'

        try: 
            if using_spei:
                with open(os.path.join(main_path, f'1lag_{distribution}_daily_{spei_spi}_{percentile}_results_data_{seed}.pkl'), 'rb') as f:
                    seed_results = pickle.load(f)
                with open(f'/gpfs/scratch/bsc32/bsc167965/data/test_train_n_shap_dilation/{site}/SHAP/1lag_{distribution}_daily_{spei_spi}_{percentile}_SHAP_values_GradientExplainer_{seed}_{number_lags}lags.pkl', 'rb') as f:
                    shap_results = pickle.load(f)
            else:
                with open(os.path.join(main_path, f'nolags_{percentile}_results_data_{seed}.pkl'), 'rb') as f:
                    seed_results = pickle.load(f)
                with open(f'/gpfs/scratch/bsc32/bsc167965/data/test_train_n_shap_dilation/{site}/SHAP/nolags_{percentile}_SHAP_values_GradientExplainer_not_filtered_{seed}.pkl', 'rb') as f:
                    shap_results = pickle.load(f)
        except FileNotFoundError:
            print(f"File not found for seed {seed}")
            continue 
            
        # Load member results
        outputs_prob_seeds[count_seed]= seed_results['out_probs_seed']
    
        shap_values_nn_raw = shap_results['nn']
        shap_values_cnn_raw = shap_results['cnn']
    
        ensamble_shap_values_nn_raw[count_seed] = shap_values_nn_raw
        ensamble_shap_values_cnn_raw[count_seed] = shap_values_cnn_raw
    
    
    output_prob_ensamble = np.mean(outputs_prob_seeds,axis=0)
    std_ensamble = np.std(outputs_prob_seeds,axis=0)
    
    # Evaluate ensemble model -----------------------------------------------------------------------------------------------
    y_true, y_pred, extreme_acc, nonextreme_acc = machine_learning.evaluate_ensamble(CombinedModel=model,cnn=CNN_model_loaded,nn=NN_model_loaded, test_loader=combined_test_loader, probs_ensamble=output_prob_ensamble, print_accuracies=True) # add extraction of output probabilities 
    
    # Mean  -----------------------------------------------------------------------------------------------------------------
    mean_ensamble_shap_values_nn_raw = np.mean(ensamble_shap_values_nn_raw,axis=0)
    mean_ensamble_shap_values_cnn_raw = np.mean(ensamble_shap_values_cnn_raw,axis=0)
    
    # Standard deviation  ---------------------------------------------------------------------------------------------------
    std_ensamble_shap_values_nn_raw = np.std(ensamble_shap_values_nn_raw,axis=0)
    std_ensamble_shap_values_cnn_raw = np.std(ensamble_shap_values_cnn_raw,axis=0)
    
    # Dictionary to save results -----------------------------------------------------------------------------------------------
    site_results = {
            'y_true_pred_pairs': (y_true,y_pred), 
            'out_probs_sites': output_prob_ensamble,
            'std_probs_sites': std_ensamble,
            'extreme_accuracy': extreme_acc,
            'nonextreme_accuracy': nonextreme_acc,
        }

    # Compute balanced accuracy
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
        
    # Prepare results text
    results_text = (
        f"Site: {site}\n"
        f"Extreme Accuracy: {extreme_acc:.4f}\n"
        f"Non-Extreme Accuracy: {nonextreme_acc:.4f}\n"
        f"Balanced Accuracy: {balanced_acc:.4f}\n"
        "--------------------------------------\n"
    )

    # ============================================================================================================================
    # Save important results to txt file
    # ============================================================================================================================

    #if using_spei:
    #    with open(f"/path/to/save/statistical/results/txt", "a") as f:  # "a" appends results for multiple sites
    #        f.write(results_text)
    #else:
    #    with open(f"/path/to/save/statistical/results/txt", "a") as f:  # "a" appends results for multiple sites
    #        f.write(results_text)
    
    #main_path = '/your/path/to/save/dictionary/ensamble_results/arrays'

    # Save dictionary results 
    #if using_spei:
    #    with open(os.path.join(main_path, 'file_name.pkl'), 'wb') as f:
    #        pickle.dump(site_results, f)
    #else:
    #    with open(os.path.join(main_path, 'file_name.pkl'), 'wb') as f:
    #        pickle.dump(site_results, f)
    
    # = ===========================================================================================================================
    # SHAP results-----------------------------------------------------------------------------------------------------------------
    # = ===========================================================================================================================
    
    shap_results = {
            'nn_mean': mean_ensamble_shap_values_nn_raw, # These are from the current site
            'nn_std': std_ensamble_shap_values_nn_raw,
            'cnn_mean': mean_ensamble_shap_values_cnn_raw,
            'cnn_std': std_ensamble_shap_values_cnn_raw
        }
    
    main_path = '/your/path/to/save/dictionary/ensamble_results/SHAP'

    #if using_spei:
    #    with open(os.path.join(main_path, 'name_file.pkl'), 'wb') as f:
    #        pickle.dump(shap_results, f)
    #else:    
    #    with open(os.path.join(main_path, 'name_file.pkl'), 'wb') as f:
    #                    pickle.dump(shap_results, f)


In [ ]:
import os
import sys
import torch
import hydra
from omegaconf import OmegaConf
from datetime import datetime
import numpy as np
import random
import pickle  # Added missing import
from pathlib import Path

# Added missing imports based on code usage
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import balanced_accuracy_score

print(f"System Path: {sys.path}")

# Assuming these are your custom modules
from quantifydrivers import machine_learning
# import convnext_functions # Uncomment if you have this module available locally

def generate_ensemble_seeds(fixed_seed=123):
    rng = np.random.default_rng(fixed_seed)
    seeds = rng.integers(low=0, high=2**32 - 1, size=20).tolist()
    return seeds

def reset_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # g.manual_seed(seed) # Passed explicitly later

print('IMPORTS DONE -------------------------------------------------')

list_seeds = generate_ensemble_seeds(fixed_seed=123)
print(f"Generated {len(list_seeds)} seeds for ensemble.")

sites = ['cordoba']
percentile ='90p'
# File paths large-scale data
file_g500 = "/gpfs/scratch/bsc32/bsc167965/data/era5/lagged_anomalies/g500_1x1_lagged_standarized_anomalies.nc"
file_g200 = "/gpfs/scratch/bsc32/bsc167965/data/era5/lagged_anomalies/g200_1x1_lagged_standarized_anomalies.nc"
file_psl = "/gpfs/scratch/bsc32/bsc167965/data/era5/lagged_anomalies/psl_1x1_lagged_standarized_anomalies.nc"
file_CO2 = "/gpfs/scratch/bsc32/bsc167965/data/daily_co2_JJA.nc"

# ========================================================================================
# Configuration
# ========================================================================================
percentile = '90p'
using_spei = False
scales_spei = ['30','60']
spei_spi = 'spei'
spei_spi_variable_mapping = {
    'spei': [f'spei_hg_{scale_spei}' for scale_spei in scales_spei],
    'spi': [f'spi_{scale_spei}' for scale_spei in scales_spei]
}
spei_variables = spei_spi_variable_mapping[spei_spi]
distribution = 'gamma'
start_date = "1950-01-01"

variables_era5 = ['g500', 'g200', 'psl']
variables_era5land = ['swvl1','swvl2','swvl3']

print(f"CONFIG | SPEI: {using_spei} | Variable: {spei_spi} | Dist: {distribution}")

if using_spei:
    _ERA5LAND_TEST_DATASET_CONF = dict(start_date="2014-01-01", end_date="2023-12-31", months=[6,7,8])
else:
    _ERA5LAND_TEST_DATASET_CONF = dict(start_date="2014-01-01", end_date="2023-12-31", months=[6,7,8], variables=variables_era5land)

_ERA5_TEST_DATASET_CONF = dict(start_date="2014-01-01", end_date="2023-12-31", months=[6,7,8], start_lag=1, lags_era5=1, variables=variables_era5)

# Start loop -------------------------------------------------------------------------------

device = torch.device("cpu") # Change to cuda if needed
print(f"Running on device: {device}")

for site in sites:
    print(f"\n{'='*40}")
    print(f"PROCESSING SITE: {site}")
    print(f"{'='*40}")

    # Local scale data files
    if spei_spi == 'spi':
        files_spei =[f"/gpfs/projects/bsc32/bsc167965/observational_TX/{distribution}_{spei_spi}_tasmax_{site}_{scale_spei}_daily.nc" for scale_spei in scales_spei]
    elif spei_spi == 'spei':
        files_spei = [f"/gpfs/projects/bsc32/bsc167965/observational_TX/{distribution}_daily_{spei_spi}{scale_spei}_tasmax_{site}_hg.nc" for scale_spei in scales_spei]

    print("Loading Large-Scale Datasets...")
    test_features_era5 = machine_learning.LargeScale_Dataset_extremes(file_g500,file_g200,file_psl, **_ERA5_TEST_DATASET_CONF)
    print(f"  -> Large-Scale Test Shape: {test_features_era5.features.shape}")

    print("Loading Local-Scale Datasets...")
    # Local-scale
    if using_spei:
        file_local_scale = f"/gpfs/scratch/bsc32/bsc167965/data/era5_land/obs_lagged_anomalies_and_event_detection/obs_{site}_lagged_standarized_anomalies_and_extreme_detection.nc"
        print("  -> using SPEI dataset (Code commented out in example as class wasn't imported)")
    else:
        file_local_scale = f"/gpfs/scratch/bsc32/bsc167965/data/era5land/lagged_anomalies_and_event_detection/{percentile}_{site}_lagged_standarized_anomalies_and_extreme_detection.nc"

        test_dataset = machine_learning.LocalScale_Dataset_extremes_location_swvl_averaged_including_CO2(
            file_path=file_local_scale, file_CO2=file_CO2, **_ERA5LAND_TEST_DATASET_CONF)

    print(f"  -> Local Test samples: {len(test_dataset)}")

    _DATALOADERS_TEST_CONF = dict(
                batch_size= 32,
                drop_last= False,
                shuffle = False,
                num_workers=0)

    combined_test_dataset = machine_learning.CombinedDataset(test_dataset,test_features_era5, variables = variables_era5  )
    combined_test_loader = DataLoader(combined_test_dataset, **_DATALOADERS_TEST_CONF)

    # Prepare for ensemble results
    outputs_prob_seeds = np.zeros((len(list_seeds), test_dataset.features.shape[0], 2))
    ensamble_shap_values_nn_raw = np.zeros((len(list_seeds), test_dataset.features.shape[0], test_dataset.features.shape[1]),dtype=np.float32)
    ensamble_shap_values_cnn_raw = np.zeros((len(list_seeds), test_dataset.features.shape[0], test_features_era5.features.shape[1], 58, 124),dtype=np.float32)

    print(f"Ensemble arrays initialized. Output shape: {outputs_prob_seeds.shape}")

    # Loop over seeds
    for count_seed, seed in enumerate(list_seeds[:]):

        main_path = f'/gpfs/scratch/bsc32/bsc214253/results/{site}/'


        try:
            if using_spei:
                file_res = os.path.join(main_path, f'1lag_{distribution}_daily_{spei_spi}_{percentile}_results_data_{seed}.pkl')
                file_shap = f'/gpfs/scratch/bsc32/bsc167965/data/test_train_n_shap_dilation/{site}/SHAP/1lag_{distribution}_daily_{spei_spi}_{percentile}_SHAP_values_GradientExplainer_{seed}_{number_lags}lags.pkl'
            else:
                file_res = os.path.join(main_path, f'{site}_{percentile}_{seed}',f'{site}_{percentile}_{seed}_evaluation.pkl')
                file_shap = os.path.join(main_path, f'{site}_{percentile}_{seed}',f'{site}_{percentile}__{seed}_shap.pkl')

            with open(file_res, 'rb') as f:
                seed_results = pickle.load(f)

            with open(file_shap, 'rb') as f:
                shap_results = pickle.load(f)

        except FileNotFoundError as e:
            print(f"    [!] File missing for seed {seed}: {e.filename}")
            continue

        outputs_prob_seeds[count_seed] = seed_results['out_probs_seed']
        ensamble_shap_values_nn_raw[count_seed] = shap_results['nn']
        ensamble_shap_values_cnn_raw[count_seed] = shap_results['cnn']

    print("Calculating Ensemble Statistics...")
    output_prob_ensamble = np.mean(outputs_prob_seeds,axis=0)
    std_ensamble = np.std(outputs_prob_seeds,axis=0)

    # Evaluate ensemble model -----------------------------------------------------------------------------------------------
    y_true, y_pred, extreme_acc, nonextreme_acc = machine_learning.evaluate_ensamble(test_loader=combined_test_loader, probs_ensamble=output_prob_ensamble, print_accuracies=True) # add extraction of output probabilities

    print("Calculating Mean Values...")
    # Mean  -----------------------------------------------------------------------------------------------------------------
    mean_ensamble_shap_values_nn_raw = np.mean(ensamble_shap_values_nn_raw,axis=0)
    mean_ensamble_shap_values_cnn_raw = np.mean(ensamble_shap_values_cnn_raw,axis=0)

    print("Calculating Standard Deviation...")
    # Standard deviation  ---------------------------------------------------------------------------------------------------
    ensamble_shap_values_nn_raw = ensamble_shap_values_nn_raw.astype(np.float32)
    ensamble_shap_values_cnn_raw = ensamble_shap_values_cnn_raw.astype(np.float32)

    std_ensamble_shap_values_nn_raw = np.std(ensamble_shap_values_nn_raw,axis=0)
    std_ensamble_shap_values_cnn_raw = np.std(ensamble_shap_values_cnn_raw,axis=0)

    print(f"Saving {site} Results...")
    # Dictionary to save results -----------------------------------------------------------------------------------------------
    site_results = {
            'y_true_pred_pairs': (y_true,y_pred),
            'out_probs_sites': output_prob_ensamble,
            'std_probs_sites': std_ensamble,
            'extreme_accuracy': extreme_acc,
            'nonextreme_accuracy': nonextreme_acc,
        }


    balanced_acc = balanced_accuracy_score(y_true, y_pred)

    print("Preparing results text...")
    # Prepare results text
    results_text = (
        f"Site: {site}\n"
        f"Extreme Accuracy: {extreme_acc:.4f}\n"
        f"Non-Extreme Accuracy: {nonextreme_acc:.4f}\n"
        f"Balanced Accuracy: {balanced_acc:.4f}\n"
        "--------------------------------------\n"
    )


    # ============================================================================================================================
    # Save important results to txt file
    # ============================================================================================================================

    if using_spei:
        with open(f"/gpfs/scratch/bsc32/bsc214253/results/txt/{percentile}_results_data_{site}.pkl", "a") as f:  # "a" appends results for multiple sites
            f.write(results_text)


    else:
        save_dir = "/gpfs/scratch/bsc32/bsc214253/results/txt"
        os.makedirs(save_dir, exist_ok=True)
        pickle.dump(site_results, open(f"{save_dir}/{percentile}_results_data_{site}.pkl", "wb"))

        file_path = f"{save_dir}/{percentile}_results_data_{site}.txt"
        with open(file_path, "a") as f:
            f.write(results_text)



    print("Preparing shap results...")
    shap_results = {
            'nn_mean': mean_ensamble_shap_values_nn_raw, # These are from the current site
            'nn_std': std_ensamble_shap_values_nn_raw,
            'cnn_mean': mean_ensamble_shap_values_cnn_raw,
            'cnn_std': std_ensamble_shap_values_cnn_raw
        }
    if using_spei:
        with open(f"/gpfs/scratch/bsc32/bsc214253/results/txt/{percentile}_shap_data_{site}.pkl", "a") as f:  # "a" appends results for multiple sites
            f.write(shap_results)




    else:
        save_dir = "/gpfs/scratch/bsc32/bsc214253/results/txt"
        with open(os.path.join(save_dir, f"{save_dir}/{percentile}_shap_data_{site}.pkl"), 'wb') as f:
                        pickle.dump(shap_results, f)


    print("RESULTS:")
    print(results_text)

print("SCRIPT COMPLETED SUCCESSFULLY")



## Loss plots ensambles

This block aggregates and visualizes the **training and validation loss curves** for the CombinedModel across multiple random seeds and sites.  

- **Input:** Loss histories (`losses_train`, `losses_val`) saved per seed and per site.  
- **Process:**  
  1. For each site, load loss curves for all seeds.  
  2. Trim sequences to the shortest run length to ensure alignment.  
  3. Compute mean and standard deviation across seeds for both training and validation.  
  4. Plot mean loss curves with shaded uncertainty bands.  
- **Outpu

In [ ]:
# Function to generate a fixed list of random seeds for reproducibility
def generate_ensemble_seeds(fixed_seed=123):
    rng = np.random.default_rng(fixed_seed)  # Create reproducible random number generator
    seeds = rng.integers(low=0, high=2**32 - 1, size=20).tolist()  # Generate 20 random seeds
    return seeds

# Generate list of seeds (same every run because of fixed_seed)
list_seeds = generate_ensemble_seeds(fixed_seed=123)

# Sites to evaluate
sites = ['cordoba','hannover','lyon' ,'stockholm','belgrado']

# Create subplot grid (2 rows x 3 columns) to plot training/validation losses per site
fig_site_loss, ax_site_loss = plt.subplots(2,3,figsize=(12,8))
ax_site_loss = ax_site_loss.flatten()  # Flatten for easy indexing

count_plot = 0  # Counter to track subplot index

# Loop over each site and aggregate results across seeds
for site in sites:

    losses_train_all_seeds = []  # Store training losses from all seeds
    losses_val_all_seeds = []    # Store validation losses from all seeds

    # Loop over seeds and load corresponding results
    for seed in list_seeds:

        # Path to the result files for this site and seed
        main_path = f'/gpfs/scratch/bsc32/bsc214253/results/{site}/'
        file_losses = os.path.join(main_path, f'{site}_{percentile}_{seed}_losses.pkl')
        main_path = f'/path/of/results/files/members'
        with open(file_losses, 'rb') as f:
            seed_results = pickle.load(f)  # Load dictionary with loss histories
    
        # Extract training and validation loss curves
        loss_train = seed_results['losses_train']
        loss_val = seed_results['losses_val']
    
        losses_train_all_seeds.append(loss_train)  
        losses_val_all_seeds.append(loss_val)

    # Ensure all loss sequences have the same length by trimming to shortest run
    min_len = min([len(loss) for loss in losses_train_all_seeds])  
    train_losses_trimmed = np.array([loss[:min_len] for loss in losses_train_all_seeds])
    val_losses_trimmed = np.array([loss[:min_len] for loss in losses_val_all_seeds])
    
    # Compute mean and standard deviation across seeds for each epoch
    mean_train = train_losses_trimmed.mean(axis=0)
    std_train = train_losses_trimmed.std(axis=0)
    
    mean_val = val_losses_trimmed.mean(axis=0)
    std_val = val_losses_trimmed.std(axis=0)
    
    # Epoch indices (1-based)
    epochs = np.arange(1, min_len + 1)

    # Plot mean + uncertainty (std band) for training and validation losses
    ax_site_loss[count_plot].plot(epochs, mean_train, label='Train', color='blue')
    ax_site_loss[count_plot].fill_between(epochs, mean_train - std_train, mean_train + std_train, color='blue', alpha=0.2)
    ax_site_loss[count_plot].plot(epochs, mean_val, label='Val', color='orange')
    ax_site_loss[count_plot].fill_between(epochs, mean_val - std_val, mean_val + std_val, color='orange', alpha=0.2)
    
    # Add labels, title, legend, and axis limits
    ax_site_loss[count_plot].set_xlabel("epoch")
    ax_site_loss[count_plot].set_ylabel("loss")
    ax_site_loss[count_plot].set_title(f"CombinedModel Loss - {site}")
    ax_site_loss[count_plot].legend()
    ax_site_loss[count_plot].set_ylim(0.2, 0.8)  # Keep consistent y-limits for better comparison

    count_plot += 1  # Move to next subplot

# Output file name for the figure
name_save_losses_fig = "losses_all_locations_CombinedModel"
    
# Save figure in the specified folder
plot_save_path = "/your/path/to/save/figures/CombinedModel_losses"
plt.tight_layout()
plt.show()
#os.makedirs(plot_save_path, exist_ok=True)  # Create directory if it does not exist
#fig_site_loss.savefig(os.path.join(plot_save_path, f"95p_{name_save_losses_fig}.png"))
#plt.close(fig_site_loss)  # Close figure to free memory


In [ ]:
import matplotlib.pyplot as plt
# Function to generate a fixed list of random seeds for reproducibility
def generate_ensemble_seeds(fixed_seed=123):
    rng = np.random.default_rng(fixed_seed)  # Create reproducible random number generator
    seeds = rng.integers(low=0, high=2**32 - 1, size=20).tolist()  # Generate 20 random seeds
    return seeds

# Generate list of seeds (same every run because of fixed_seed)
list_seeds = generate_ensemble_seeds(fixed_seed=123)

# Sites to evaluate
sites = ['cordoba']

# Create subplot grid (2 rows x 3 columns) to plot training/validation losses per site
fig_site_loss, ax_site_loss = plt.subplots(2,3,figsize=(12,8))
ax_site_loss = ax_site_loss.flatten()  # Flatten for easy indexing

count_plot = 0  # Counter to track subplot index

# Loop over each site and aggregate results across seeds
for site in sites:

    losses_train_all_seeds = []  # Store training losses from all seeds
    losses_val_all_seeds = []    # Store validation losses from all seeds

    # Loop over seeds and load corresponding results
    for seed in list_seeds:

        # Path to the result files for this site and seed
        main_path = f'/gpfs/scratch/bsc32/bsc214253/results/{site}/'
        file_losses = os.path.join(main_path, f'{site}_{percentile}_{seed}',f'{site}_{percentile}_{seed}_losses.pkl')
        with open(file_losses, 'rb') as f:
            seed_results = pickle.load(f)  # Load dictionary with loss histories

        # Extract training and validation loss curves
        loss_train = seed_results['losses_train']
        loss_val = seed_results['losses_val']

        losses_train_all_seeds.append(loss_train)
        losses_val_all_seeds.append(loss_val)

    # Ensure all loss sequences have the same length by trimming to shortest run
    min_len = min([len(loss) for loss in losses_train_all_seeds])
    train_losses_trimmed = np.array([loss[:min_len] for loss in losses_train_all_seeds])
    val_losses_trimmed = np.array([loss[:min_len] for loss in losses_val_all_seeds])

    # Compute mean and standard deviation across seeds for each epoch
    mean_train = train_losses_trimmed.mean(axis=0)
    std_train = train_losses_trimmed.std(axis=0)

    mean_val = val_losses_trimmed.mean(axis=0)
    std_val = val_losses_trimmed.std(axis=0)

    # Epoch indices (1-based)
    epochs = np.arange(1, min_len + 1)

    # Plot mean + uncertainty (std band) for training and validation losses
    ax_site_loss[count_plot].plot(epochs, mean_train, label='Train', color='blue')
    ax_site_loss[count_plot].fill_between(epochs, mean_train - std_train, mean_train + std_train, color='blue', alpha=0.2)
    ax_site_loss[count_plot].plot(epochs, mean_val, label='Val', color='orange')
    ax_site_loss[count_plot].fill_between(epochs, mean_val - std_val, mean_val + std_val, color='orange', alpha=0.2)

    # Add labels, title, legend, and axis limits
    ax_site_loss[count_plot].set_xlabel("epoch")
    ax_site_loss[count_plot].set_ylabel("loss")
    ax_site_loss[count_plot].set_title(f"CombinedModel Loss - {site}")
    ax_site_loss[count_plot].legend()
    ax_site_loss[count_plot].set_ylim(0.2, 0.8)  # Keep consistent y-limits for better comparison

    count_plot += 1  # Move to next subplot

# Output file name for the figure
name_save_losses_fig = "losses_all_locations_CombinedModel"

# Save figure in the specified folder
plot_save_path = "/gpfs/scratch/bsc32/bsc214253/plots/CombinedModel_losses"
plt.tight_layout()
plt.show()
os.makedirs(plot_save_path, exist_ok=True)  # Create directory if it does not exist
fig_site_loss.savefig(os.path.join(plot_save_path, f"95p_{name_save_losses_fig}.png"))
plt.close(fig_site_loss)  # Close figure to free memory
